> [!WARNING]
> **OFFLINE SUBMISSION NOTEBOOK**
> Internet is disabled. Upload the following as a Kaggle dataset:
> 1. `b2-efficientnet-checkpoints` â€” fold checkpoints (`best_model_fold0.pth` ... `best_model_fold4.pth`)

# B2: EfficientNet-B3 â€” 5-Fold Ensemble Inference

**Author:** Mridankan Mandal

**Competition:** [CSIRO Image2Biomass](https://www.kaggle.com/competitions/csiro-biomass)

---

## Overview

Inference notebook for EfficientNet-B3 (ImageNet pretrained backbone, fine-tuned locally).
5-fold ensemble with weighted MSE loss and simple MLP head.
Uses single full-image input (not split into left/right).

## Inference and Submission Generation (B2)

In [ ]:
import os, gc, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from torchvision import transforms
from torchvision.models import efficientnet_b3

print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
import glob as _glob
def _find(slug, pattern=None):
    """Find Kaggle dataset dir. If pattern given, searches subdirs too."""
    for base in [f'/kaggle/input/{slug}', *_glob.glob(f'/kaggle/input/datasets/*/{slug}')]:
        if not os.path.isdir(base): continue
        if pattern:
            if _glob.glob(os.path.join(base, pattern)): return base
            for sub in _glob.glob(os.path.join(base, '*')):
                if os.path.isdir(sub) and _glob.glob(os.path.join(sub, pattern)):
                    return sub
        return base
    raise FileNotFoundError(f"Dataset '{slug}' not found in /kaggle/input/")


In [ ]:
class CFG:
    BASE_PATH = '/kaggle/input/competitions/csiro-biomass'
    TEST_CSV = os.path.join(BASE_PATH, 'test.csv')
    TEST_IMAGE_DIR = os.path.join(BASE_PATH, 'test')
    MODEL_DIR = _find('b2-efficientnet-checkpoints', '*.pth')
    N_FOLDS = 5
    FOLDS_TO_USE = [0, 1, 2, 3, 4]
    IMG_SIZE = 384
    BATCH_SIZE = 16
    NUM_WORKERS = 0
    TARGET_COLS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Device: {CFG.DEVICE}, Checkpoints: {CFG.MODEL_DIR}")

In [ ]:
# --- Model Architecture (must match training exactly) ---
class BiomassModel(nn.Module):
    def __init__(self, num_targets=5):
        super().__init__()
        self.backbone = efficientnet_b3(weights=None)  # no pretrained weights at inference
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()
        self.head = nn.Sequential(
            nn.Linear(in_features, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, num_targets)
        )
    def forward(self, x):
        return self.head(self.backbone(x))

print("Model architecture defined")

In [ ]:
# --- Test Data & Inference ---
val_tfm = transforms.Compose([
    transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class TestBiomassDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.paths = df['image_path'].values
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        path = os.path.join(self.img_dir, os.path.basename(self.paths[idx]))
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img

# Load test data
test_long = pd.read_csv(CFG.TEST_CSV)
test_long['image_id'] = test_long['sample_id'].str.split('__').str[0]
test_df = test_long.drop_duplicates('image_id')[['image_id','image_path']].reset_index(drop=True)
print(f"Test images: {len(test_df)}")

test_dataset = TestBiomassDataset(test_df, CFG.TEST_IMAGE_DIR, val_tfm)
test_loader = DataLoader(test_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False,
                         num_workers=CFG.NUM_WORKERS, pin_memory=True)

@torch.no_grad()
def predict_test(model, loader, device):
    model.eval(); all_preds = []
    for imgs in tqdm(loader, desc='Inference'):
        with autocast('cuda'):
            preds = model(imgs.to(device))
        all_preds.append(preds.cpu().numpy())
    return np.concatenate(all_preds)

# Ensemble across folds
all_fold_preds = []
for fold in CFG.FOLDS_TO_USE:
    ckpt_path = os.path.join(CFG.MODEL_DIR, f'best_model_fold{fold}.pth')
    if not os.path.exists(ckpt_path):
        print(f'Checkpoint not found: {ckpt_path}, skipping fold {fold}.')
        continue
    model = BiomassModel().to(CFG.DEVICE)
    model.load_state_dict(torch.load(ckpt_path, map_location=CFG.DEVICE, weights_only=True))
    print(f'Loaded fold {fold}')
    all_fold_preds.append(predict_test(model, test_loader, CFG.DEVICE))
    del model; gc.collect(); torch.cuda.empty_cache()

if not all_fold_preds:
    raise RuntimeError('No fold checkpoints found.')
avg_preds = np.mean(all_fold_preds, axis=0)
print(f'Ensemble of {len(all_fold_preds)} folds, shape: {avg_preds.shape}')

In [ ]:
# --- Submission ---
image_ids = test_df['image_id'].values
pred_map = {}
for i, img_id in enumerate(image_ids):
    for j, tn in enumerate(CFG.TARGET_COLS):
        pred_map[(img_id, tn)] = float(avg_preds[i, j])

test_long['target'] = test_long.apply(
    lambda row: pred_map.get((row['image_id'], row['target_name']), 0.0), axis=1)
df_sub = test_long[['sample_id','target']].copy()

sample_sub = pd.read_csv(os.path.join(CFG.BASE_PATH, 'sample_submission.csv'))
df_sub = sample_sub[['sample_id']].merge(df_sub, on='sample_id', how='left')
df_sub['target'] = df_sub['target'].fillna(0.0)
df_sub.to_csv('submission.csv', index=False)
print(f'Saved submission.csv, shape: {df_sub.shape}')
print(df_sub.head(10).to_string(index=False))